# CrewAI LLM Connections [Production - Module 04]

> **MLCourse - Agentic AI - CrewAI Production**

CrewAI connects to LLMs through LiteLLM, which provides a unified interface
for Ollama, Groq, OpenAI, and dozens of other providers. This module covers
LiteLLM provider configuration, Ollama local models, Groq cloud (guarded),
model fallback strategies, temperature/max_tokens configuration, and the
LLMSelection strategy pattern.

## What you will learn

1. How CrewAI uses LiteLLM as the LLM abstraction layer.
2. Configuring Ollama for local, free inference.
3. Configuring Groq for fast cloud inference (API key guarded).
4. Configuring OpenAI for high-quality cloud inference (API key guarded).
5. Model fallback: automatic switching when a provider is unavailable.
6. Temperature and max_tokens configuration per agent.
7. LLMSelection strategy for dynamic provider selection.

## Key takeaways

- All LLM configs go through the `LLM` class from CrewAI.
- Ollama requires no API key and runs locally.
- Groq and OpenAI require API keys loaded from `.env`.
- Fallback chains ensure your crew runs even if one provider is down.
- Different agents in the same crew can use different LLM providers.

In [ ]:
# ---- Setup: imports, environment, track discovery ---------------------------

import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
load_dotenv(TRACK / ".env", override=False)

# Check which API keys are available.
OPENAI_KEY = os.environ.get("OPENAI_API_KEY", "")
GROQ_KEY = os.environ.get("GROQ_API_KEY", "")

print("API Key Status:")
print(f"  OPENAI_API_KEY: {'FOUND' if OPENAI_KEY else 'NOT SET'}")
print(f"  GROQ_API_KEY:   {'FOUND' if GROQ_KEY else 'NOT SET'}")
print(f"  Ollama:         Always available (local, no key needed)")

In [ ]:
# ---- Check Ollama availability --------------------------------------------

OLLAMA_OK = False
try:
    from langchain_ollama import ChatOllama
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    OLLAMA_OK = True
    print("\nOllama: ONLINE (llama3.1:8b)")
except Exception as e:
    print(f"\nOllama: OFFLINE -- {e}")

In [ ]:
# ---- CrewAI imports ---------------------------------------------------------

try:
    from crewai import Agent, Task, Crew, Process, LLM
    CREWAI_OK = True
    print("CrewAI version:", __import__("crewai").__version__)
except ImportError as e:
    CREWAI_OK = False
    print("CrewAI not installed:", e)

## 1. LiteLLM -- The Universal LLM Interface

CrewAI uses LiteLLM under the hood. LiteLLM provides a single `completion()`
API that translates to 100+ LLM providers. In CrewAI, you configure the
LLM via the `LLM` class with a model string.

Model string format: `provider/model_name`
- `ollama/llama3.2` -- Ollama local model
- `groq/llama-3.3-70b-versatile` -- Groq cloud
- `openai/gpt-4o` -- OpenAI cloud
- `anthropic/claude-3-5-sonnet` -- Anthropic cloud

LiteLLM handles authentication, retries, and rate limiting automatically.

In [ ]:
# ---- LiteLLM provider reference -------------------------------------------

print("=== LiteLLM Provider Reference ===\n")
providers = [
    ("Ollama (local)", "ollama/llama3.1:8b", "No key needed"),
    ("Groq (cloud)", "groq/llama-3.3-70b-versatile", "GROQ_API_KEY"),
    ("OpenAI (cloud)", "openai/gpt-4o", "OPENAI_API_KEY"),
    ("Anthropic (cloud)", "anthropic/claude-3-5-sonnet", "ANTHROPIC_API_KEY"),
    ("HuggingFace", "huggingface/meta-llama/Llama-3-70b", "HF token"),
    ("Azure OpenAI", "azure/gpt-4o", "Azure credentials"),
    ("Together AI", "together/meta-llama/Llama-3-70b", "TOGETHER_API_KEY"),
]
for name, model, auth in providers:
    print(f"  {name:20s} {model:45s} Auth: {auth}")

## 2. Provider 1: Ollama (Local, Free)

Ollama runs models locally on your machine. No API key, no cost, no rate
limits. The trade-off is speed and model quality depend on your hardware.

Configuration:
```python
LLM(model="ollama/llama3.2", temperature=0.7)
```

Prerequisites:
- Ollama installed and running (`ollama serve`).
- Model pulled (`ollama pull llama3.2`).

In [ ]:
# ---- Provider 1: Ollama configuration ------------------------------------

print("=== Provider 1: Ollama (Local) ===\n")

if CREWAI_OK and OLLAMA_OK:
    ollama_llm = LLM(
        model="ollama/llama3.1:8b",
        temperature=0.7,
        # base_url can be set if Ollama runs on a non-default host.
        # base_url="http://localhost:11434",
    )
    print(f"Ollama LLM configured:")
    print(f"  Model: {ollama_llm.model}")
    print(f"  Temperature: {ollama_llm.temperature}")
    print(f"  API key: Not needed (local)")
    print()

    # Quick test.
    result = ollama_llm.call("Say 'Ollama works' in exactly 3 words.")
    print(f"Test response: {result}")
else:
    print("[SKIP] Ollama not available")

## 3. Provider 2: Groq (Fast Cloud Inference)

Groq provides extremely fast inference for open-source models. It requires
a `GROQ_API_KEY` but offers a generous free tier.

Configuration:
```python
LLM(model="groq/llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])
```

Note: Groq model names differ from Ollama model names.

In [ ]:
# ---- Provider 2: Groq configuration (guarded) -----------------------------

print("=== Provider 2: Groq (Cloud, Fast) ===\n")

GROQ_OK = False
if CREWAI_OK and GROQ_KEY:
    try:
        groq_llm = LLM(
            model="groq/llama-3.3-70b-versatile",
            api_key=GROQ_KEY,
            temperature=0.7,
        )
        result = groq_llm.call("Say 'Groq works' in exactly 3 words.")
        print(f"Groq LLM configured and tested:")
        print(f"  Model: {groq_llm.model}")
        print(f"  Temperature: {groq_llm.temperature}")
        print(f"  Test response: {result}")
        GROQ_OK = True
    except Exception as e:
        print(f"Groq connection failed: {e}")
        print("Check your GROQ_API_KEY in .env")
elif CREWAI_OK:
    print("Groq LLM configuration (API key required):")
    print('  groq_llm = LLM(model="groq/llama-3.3-70b-versatile", api_key=GROQ_KEY)')
    print()
    print("To enable Groq:")
    print("  1. Get a free key at https://console.groq.com")
    print("  2. Add GROQ_API_KEY=your_key to 03_agentic_ai/.env")
else:
    print("[SKIP] CrewAI not installed")

## 4. Provider 3: OpenAI (High Quality Cloud)

OpenAI provides GPT-4o and other high-quality models. Requires `OPENAI_API_KEY`.

Configuration:
```python
LLM(model="openai/gpt-4o", api_key=os.environ["OPENAI_API_KEY"])
```

Note: OpenAI models are the most expensive but often the highest quality.

In [ ]:
# ---- Provider 3: OpenAI configuration (guarded) --------------------------

print("=== Provider 3: OpenAI (Cloud, High Quality) ===\n")

OPENAI_OK = False
if CREWAI_OK and OPENAI_KEY:
    try:
        openai_llm = LLM(
            model="openai/gpt-4o",
            api_key=OPENAI_KEY,
            temperature=0.7,
        )
        # Don't actually call -- just show configuration.
        print(f"OpenAI LLM configured:")
        print(f"  Model: {openai_llm.model}")
        print(f"  Temperature: {openai_llm.temperature}")
        print(f"  API key: SET (length={len(OPENAI_KEY)})")
        OPENAI_OK = True
    except Exception as e:
        print(f"OpenAI configuration failed: {e}")
elif CREWAI_OK:
    print("OpenAI LLM configuration (API key required):")
    print('  openai_llm = LLM(model="openai/gpt-4o", api_key=OPENAI_KEY)')
    print()
    print("To enable OpenAI:")
    print("  1. Get a key at https://platform.openai.com/api-keys")
    print("  2. Add OPENAI_API_KEY=your_key to 03_agentic_ai/.env")
else:
    print("[SKIP] CrewAI not installed")

## 5. Side-by-Side Provider Comparison

| Feature | Ollama | Groq | OpenAI |
|---------|--------|------|--------|
| Cost | Free | Free tier + paid | Paid |
| Speed | Hardware-dependent | Very fast | Fast |
| Quality | Good (7B-70B models) | Great (70B models) | Best (GPT-4o) |
| Privacy | Fully local | Cloud | Cloud |
| Rate Limits | None | Generous | Moderate |
| Setup | Install Ollama | API key only | API key only |
| Model Names | `ollama/llama3.2` | `groq/llama-3.3-70b-versatile` | `openai/gpt-4o` |

In [ ]:
# ---- Side-by-side comparison table ----------------------------------------

print("=== Provider Comparison ===\n")
print(f"{'Feature':20s} {'Ollama':20s} {'Groq':20s} {'OpenAI':20s}")
print("-" * 80)
rows = [
    ("Cost", "Free", "Free tier + paid", "Paid"),
    ("Speed", "Hardware-dependent", "Very fast", "Fast"),
    ("Quality", "Good (7B-70B)", "Great (70B)", "Best (GPT-4o)"),
    ("Privacy", "Fully local", "Cloud", "Cloud"),
    ("Rate Limits", "None", "Generous", "Moderate"),
    ("Model", "ollama/llama3.1:8b", "groq/llama-3.3-70b", "openai/gpt-4o"),
]
for row in rows:
    print(f"{row[0]:20s} {row[1]:20s} {row[2]:20s} {row[3]:20s}")

## 6. Model Fallback Strategy

In production, LLM providers can go down. A fallback chain ensures your
crew continues working by trying providers in order of preference.

In [ ]:
# ---- Model fallback implementation ----------------------------------------

if CREWAI_OK:
    def create_llm_with_fallback():
        """Try providers in order: Ollama -> Groq -> OpenAI.

        Returns the first available LLM configuration.
        """
        # Priority 1: Ollama (local, free, always available if running).
        if OLLAMA_OK:
            print("Fallback: Using Ollama (priority 1)")
            return LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.7)

        # Priority 2: Groq (fast, free tier).
        if GROQ_KEY:
            try:
                llm = LLM(model="groq/llama-3.3-70b-versatile",
                           api_key=GROQ_KEY, temperature=0.7)
                print("Fallback: Using Groq (priority 2)")
                return llm
            except Exception:
                pass

        # Priority 3: OpenAI (high quality).
        if OPENAI_KEY:
            try:
                llm = LLM(model="openai/gpt-4o",
                           api_key=OPENAI_KEY, temperature=0.7)
                print("Fallback: Using OpenAI (priority 3)")
                return llm
            except Exception:
                pass

        # No provider available -- return Ollama config (will fail gracefully).
        print("Fallback: No provider available, returning Ollama config")
        return LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.7)

    fallback_llm = create_llm_with_fallback()
    print(f"Selected model: {fallback_llm.model}")
else:
    print("[SKIP] CrewAI not installed")

## 7. Temperature and max_tokens Configuration

Temperature controls randomness (0 = deterministic, 1 = creative).
max_tokens limits the response length. Different tasks need different settings.

- Analysis/extraction: temperature=0, max_tokens=1000
- Creative writing: temperature=0.9, max_tokens=2000
- Code generation: temperature=0.2, max_tokens=3000
- Quick classification: temperature=0, max_tokens=100

In [ ]:
# ---- Temperature and max_tokens presets -----------------------------------

if CREWAI_OK:
    print("=== Temperature and max_tokens Presets ===\n")

    presets = {
        "analysis": {"temperature": 0.0, "max_tokens": 1000,
                     "use_case": "Data analysis, extraction, classification"},
        "creative": {"temperature": 0.9, "max_tokens": 2000,
                     "use_case": "Writing, brainstorming, content generation"},
        "code": {"temperature": 0.2, "max_tokens": 3000,
                 "use_case": "Code generation, debugging, refactoring"},
        "quick": {"temperature": 0.0, "max_tokens": 100,
                  "use_case": "Classification, routing, quick decisions"},
        "balanced": {"temperature": 0.5, "max_tokens": 1500,
                     "use_case": "General purpose, chat, Q&A"},
    }

    for name, config in presets.items():
        print(f"  {name:12s}: temp={config['temperature']}, "
              f"max_tokens={config['max_tokens']:5d}  -- {config['use_case']}")

    print()
    print("Example per-agent configuration:")
    print("  analyst = Agent(role='Analyst', llm=LLM('ollama/llama3.1:8b', temperature=0))")
    print("  writer  = Agent(role='Writer',  llm=LLM('ollama/llama3.1:8b', temperature=0.9))")

## 8. Per-Agent LLM Configuration

Different agents in the same crew can use different LLMs. This is powerful:
use a fast, cheap model for simple tasks and a powerful model for complex ones.

In [ ]:
# ---- Per-agent LLM configuration example ---------------------------------

if CREWAI_OK and OLLAMA_OK:
    # Different agents with different LLM configurations.
    analyst_llm = LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.0)
    writer_llm = LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.9)
    coder_llm = LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.2)

    analyst = Agent(
        role="Data Analyst",
        goal="Analyze data precisely with deterministic output.",
        backstory="You are a precise data analyst.",
        llm=analyst_llm,
        verbose=False,
        allow_delegation=False,
    )

    writer = Agent(
        role="Content Writer",
        goal="Write engaging, creative content.",
        backstory="You are a creative writer.",
        llm=writer_llm,
        verbose=False,
        allow_delegation=False,
    )

    coder = Agent(
        role="Code Developer",
        goal="Write clean, efficient code.",
        backstory="You are an expert programmer.",
        llm=coder_llm,
        verbose=False,
        allow_delegation=False,
    )

    print("=== Per-Agent LLM Configuration ===\n")
    for agent in [analyst, writer, coder]:
        print(f"  {agent.role:20s}: temp={agent.llm.temperature}")

    print()
    print("All three agents are in the same crew but use different temperatures.")
    print("This lets you optimize cost and quality per task type.")

## 9. LLMSelection Strategy Pattern

For dynamic provider selection based on task requirements, implement an
LLMSelection strategy. This pattern evaluates the task and picks the
best provider at runtime.

In [ ]:
# ---- LLMSelection strategy pattern ----------------------------------------

if CREWAI_OK:
    class LLMSelectionStrategy:
        """Dynamic LLM selection based on task characteristics.

        Evaluates task complexity and picks the optimal provider.
        """

        def __init__(self):
            self.providers = []
            if OLLAMA_OK:
                self.providers.append({
                    "name": "ollama",
                    "model": "ollama/llama3.1:8b",
                    "cost_per_1k": 0.0,
                    "quality_score": 7,
                    "speed_score": 6,
                })
            if GROQ_KEY:
                self.providers.append({
                    "name": "groq",
                    "model": "groq/llama-3.3-70b-versatile",
                    "cost_per_1k": 0.0005,
                    "quality_score": 8,
                    "speed_score": 10,
                })
            if OPENAI_KEY:
                self.providers.append({
                    "name": "openai",
                    "model": "openai/gpt-4o",
                    "cost_per_1k": 0.005,
                    "quality_score": 10,
                    "speed_score": 8,
                })

        def select(self, task_complexity="medium", prefer_speed=False):
            """Select the best provider for the given task.

            Args:
                task_complexity: low, medium, or high.
                prefer_speed: If True, prioritize speed over quality.

            Returns:
                LLM instance configured with the selected provider.
            """
            if not self.providers:
                return LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.7)

            # Score each provider based on task requirements.
            scored = []
            for p in self.providers:
                if task_complexity == "high":
                    score = p["quality_score"] * 2 - p["cost_per_1k"] * 100
                elif prefer_speed:
                    score = p["speed_score"] * 2 - p["cost_per_1k"] * 100
                else:
                    score = p["quality_score"] + p["speed_score"] - p["cost_per_1k"] * 100
                scored.append((score, p))

            scored.sort(reverse=True)
            best = scored[0][1]
            return LLM(model=best["model"], temperature=0.7)

    # Demo the strategy.
    strategy = LLMSelectionStrategy()
    print("=== LLMSelection Strategy ===\n")
    print(f"Available providers: {[p['name'] for p in strategy.providers]}")
    print()

    for complexity in ["low", "medium", "high"]:
        llm = strategy.select(task_complexity=complexity)
        print(f"  Task complexity={complexity:8s} -> {llm.model}")

    print()
    llm = strategy.select(task_complexity="medium", prefer_speed=True)
    print(f"  Speed preference   -> {llm.model}")

## 10. Production LLM Configuration Checklist

Before deploying with LLM connections:

- [ ] Primary provider configured and tested.
- [ ] Fallback provider configured.
- [ ] API keys loaded from `.env` (never hardcoded).
- [ ] Temperature set appropriately per agent.
- [ ] max_tokens set to prevent runaway responses.
- [ ] Rate limit handling in place (LiteLLM handles retries).
- [ ] Cost monitoring enabled for cloud providers.
- [ ] Latency logging for performance tracking.

In [ ]:
# ---- Production checklist -------------------------------------------------

print("=== Production LLM Configuration Checklist ===\n")
checklist = [
    ("Primary provider", "Configured and tested"),
    ("Fallback provider", "Configured for failover"),
    ("API keys", "Loaded from .env, never hardcoded"),
    ("Temperature", "Set per agent (0 for analysis, 0.9 for creative)"),
    ("max_tokens", "Set to prevent runaway responses"),
    ("Rate limiting", "LiteLLM handles retries automatically"),
    ("Cost monitoring", "Track token usage per provider"),
    ("Latency logging", "Log response times for SLA compliance"),
]
for item, detail in checklist:
    print(f"  [x] {item:25s} -- {detail}")

## Summary

This notebook covered all LLM connection patterns for CrewAI:

1. **LiteLLM abstraction** -- unified interface for 100+ providers.
2. **Ollama** -- local, free, no API key needed.
3. **Groq** -- fast cloud inference with free tier.
4. **OpenAI** -- highest quality cloud models.
5. **Fallback chains** -- automatic provider switching on failure.
6. **Temperature/max_tokens** -- per-agent configuration for different tasks.
7. **LLMSelection strategy** -- dynamic provider selection at runtime.

## Next steps

- Set up a Groq API key for fast cloud inference.
- Experiment with different temperatures for your use case.
- Build a custom LLMSelection strategy for your production workload.